[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/carlospi2000/Notes_in_Computational_Physics/blob/main/1.Introduction_to_programming/Semana_01_Introduccion_Fortran_revised.ipynb)

# Conceptos Fundamentales de Programación (Fortran)
**Física Computacional — 106018C** - *Introducción a conceptos fundamentales de programación*

Este notebook es autocontenido: no necesitas material externo para seguirlo. Al final se citan secciones del libro de texto del curso (Chapman, *Fortran for Scientists and Engineers*) por si quieres profundizar, pero no son necesarias para completar la sesión ni la tarea.

## Objetivos de la sesión

- Definir qué es un algoritmo y reconocer sus características.
- Entender el proceso completo de diseño de un programa, de la idea al código funcionando.
- Diseñar algoritmos con pseudocódigo y diagramas de flujo, y entender para qué sirven.
- Reconocer la estructura de un programa Fortran (declaración, ejecución, terminación) y las convenciones de estilo.
- Declarar y usar variables, constantes y tipos de datos en Fortran, documentándolos con un diccionario de datos.
- Realizar operaciones aritméticas y lógicas, entendiendo errores comunes.
- Leer y escribir datos con `read` y `print`, resolviendo un problema físico real de principio a fin.
- Compilar, ejecutar, depurar y **corregir** programas simples en Fortran.
- Adoptar buenas prácticas de programación desde el primer programa.

## Contenido

1. ¿Qué es programar? ¿Qué es un algoritmo?
2. El proceso de diseño de un programa
3. Diagramas de flujo y pseudocódigo
4. Estructura de un programa Fortran
5. Variables, constantes y tipos de datos
6. Operaciones aritméticas
7. Funciones intrínsecas
8. Entrada y salida de datos
9. Ejemplo integrador: datación por carbono-14
10. Operadores lógicos y decisiones (`if`)
11. Depuración: cuando el programa no hace lo que esperabas
12. Buenas prácticas de programación — síntesis
13. Hoja de referencia rápida
14. Tarea
15. Para profundizar (opcional)

**Nota de alcance:** la representación binaria de los datos, la precisión y el rango de `integer`/`real`, y el error de redondeo se ven con detalle en el notebook de la **Semana 2** (donde el cronograma los ubica: "números pequeños y grandes" y "error, precisión y propagación"). Aquí solo lo mencionamos lo justo para entender por qué Fortran distingue tipos de datos.

**Estructura de la clase:** 2 horas de teoría/práctica con este notebook + 2 horas de aplicación con un problema de física (lo trae el docente aparte).

## 0. Preparar el entorno

Este notebook usa un **kernel de Python**, pero el código que escribimos, compilamos y corremos es **Fortran**: cada programa se guarda a un archivo `.f90` con la instrucción `%%writefile`, se compila con `gfortran` (`!gfortran archivo.f90 -o ejecutable`) y se corre (`!./ejecutable`), todo desde celdas de código.

- **Google Colab:** descomenta y corre la línea de abajo una vez, al inicio de la sesión.
- **Local (Linux/Mac):** necesitas `gfortran` instalado (`sudo apt install gfortran` o `brew install gcc`). En Windows, usa WSL.
- **Sin instalar nada:** [Compiler Explorer](https://godbolt.org/) u [OnlineGDB](https://www.onlinegdb.com/online_fortran_compiler) compilan Fortran en el navegador.

In [4]:
# !apt-get -qq install gfortran   # <- descomentar solo en Google Colab

!gfortran --version

GNU Fortran (Homebrew GCC 15.2.0_1) 15.2.0
Copyright (C) 2025 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.



## 1. ¿Qué es programar? ¿Qué es un algoritmo?

**Programar** es dar instrucciones a un computador, en un lenguaje que este pueda entender, para que resuelva un problema. Un computador es una máquina que almacena información y realiza cálculos sobre ella mucho más rápido de lo que un humano puede pensar — pero **no "piensa"**: solo sigue, al pie de la letra, los pasos que un programa le indica. Cuando un computador parece hacer algo inteligente, es porque una persona escribió un programa inteligente.

Detrás de cualquier programa —sin importar el lenguaje— hay siempre los mismos ingredientes: **variables** que almacenan y manipulan datos, **entrada y salida** que comunican el programa con el usuario, **operaciones** matemáticas y lógicas que transforman esos datos, y una secuencia ordenada de instrucciones —el **código**— que el computador ejecuta paso a paso.

Antes de llegar al código, hay un concepto más fundamental: el **algoritmo**.

<div style="background-color:#eef7ee; border-left:5px solid #4caf50; padding:10px 15px; margin:10px 0;">
<b>Un algoritmo:</b> es la secuencia ordenada y finita de pasos que resuelve un problema, independientemente del lenguaje de programación en que luego se implemente.
</div>

Para que una secuencia de pasos merezca llamarse algoritmo, debe cumplir:

1. **Finitud** — debe terminar después de un número finito de pasos.
2. **Precisión** — cada paso debe estar definido sin ambigüedad.
3. **Entrada** — recibe cero o más valores de entrada.
4. **Salida** — produce uno o más resultados.
5. **Efectividad** — cada paso debe ser lo bastante básico como para poder ejecutarse, en principio, a mano con lápiz y papel.

Programar es, entonces, **traducir un algoritmo a un lenguaje que el computador entiende** — pero el algoritmo debe existir *antes* de esa traducción (sección 2).

### El computador, en tres piezas

Todo computador tiene una ***CPU*** (unidad central de procesamiento: ejecuta instrucciones y hace los cálculos), una ***memoria*** (guarda temporalmente el programa y sus variables mientras corre — se divide en memoria caché, principal y secundaria, de más rápida/cara a más lenta/barata) y ***dispositivos de entrada/salida*** (teclado, mouse, pantalla, disco — por donde entran y salen los datos).

Internamente, la memoria del computador está hecha de interruptores que solo pueden estar en dos estados (encendido/apagado), representados como 1 y 0: el sistema **binario**. Por eso un entero y un número decimal *no se guardan de la misma forma en memoria*, y todo lenguaje —Fortran incluido— obliga a declarar el **tipo** de cada variable: le dice al computador cuánta memoria reservar y cómo interpretar esos bits. *(En la Semana 2 veremos esto en detalle: cuántos bits ocupa cada tipo, qué tan grande puede ser un número, y qué es el error de redondeo.)*

### De instrucciones binarias a lenguajes de alto nivel

Un computador, en el fondo, solo ejecuta cadenas de operaciones binarias muy simples (sumar, cargar un valor, comparar) llamado **lenguaje de máquina** — prácticamente imposible de escribir a mano de forma productiva. Por eso usamos **lenguajes de alto nivel** (Fortran, C++, Python, Java...): escribimos instrucciones parecidas al lenguaje natural o a ecuaciones algebraicas, y un programa llamado **compilador** las traduce al lenguaje de máquina que el computador realmente ejecuta.

### ¿Por qué Fortran?

Fortran (*FOR*mula *TRAN*slation) nació en 1957 en IBM — es, literalmente, el abuelo de los lenguajes científicos: fue el primero en permitir escribir un algoritmo como una serie de ecuaciones algebraicas, en vez de instrucciones de máquina escritas a mano. Sigue vigente hoy porque es extremadamente eficiente para cálculo numérico intensivo (clima, dinámica de fluidos, física de partículas, álgebra lineal a gran escala) y es el lenguaje dominante en supercómputo.

El lenguaje ha evolucionado enormemente. Estas tres versiones del **mismo programa** —resolver las raíces de $ax^2+bx+c=0$— muestran la diferencia entre el Fortran de 1957 y el Fortran moderno que usaremos en este curso (no las vamos a compilar, son solo para comparar):

**FORTRAN I (1957)** — sin nombres de variable descriptivos, sin `if`/`else` estructurado, formato de columnas fijas:

```fortran
C     SOLVE QUADRATIC EQUATION IN FORTRAN I
      READ 100,A,B,C
100   FORMAT(3F12.4)
      DISCR = B**2-4*A*C
      IF (DISCR) 10,20,30
10    X1=(-B)/(2.*A)
      X2=SQRTF(ABSF(DISCR))/(2.*A)
      PRINT 110,X1,X2
110   FORMAT(5H X = ,F12.3,4H +i ,F12.3)
      GOTO 40
20    X1=(-B)/(2.*A)
      PRINT 130,X1
130   FORMAT(11H X1 = X2 = ,F12.3)
      GOTO 40
30    X1=((-B)+SQRTF(ABSF(DISCR)))/(2.*A)
      X2=((-B)-SQRTF(ABSF(DISCR)))/(2.*A)
      PRINT 140,X1
140   FORMAT(6H X1 = ,F12.3)
40    CONTINUE
      STOP
```

**Fortran moderno (el que usaremos)** — formato libre, nombres descriptivos, `if`/`else if`/`else` estructurado, comentarios explicativos:

```fortran
program roots
    ! Resuelve las raices de A*X**2 + B*X + C = 0
    implicit none
    real :: a, b, c, discriminante, x1, x2, parte_real, parte_imag

    print *, "Ingrese los coeficientes A, B y C:"
    read *, a, b, c

    discriminante = b**2 - 4.0 * a * c

    if (discriminante > 0.0) then
        x1 = (-b + sqrt(discriminante)) / (2.0 * a)
        x2 = (-b - sqrt(discriminante)) / (2.0 * a)
        print *, "Dos raices reales: ", x1, x2
    else if (discriminante == 0.0) then
        x1 = -b / (2.0 * a)
        print *, "Raiz real repetida: ", x1
    else
        parte_real = -b / (2.0 * a)
        parte_imag = sqrt(abs(discriminante)) / (2.0 * a)
        print *, "Raices complejas: ", parte_real, " +-i", parte_imag
    end if

end program roots
```

Nota cuánto más fácil es entender qué hace la segunda versión sin ningún comentario adicional — esa legibilidad es, en gran parte, el resultado de siete décadas de buenas prácticas de programación acumuladas, muchas de las cuales veremos hoy.

## 2. El proceso de diseño de un programa

Un error frecuente al empezar a programar es abrir el editor y escribir código de inmediato, sin haber pensado antes en la solución. El resultado casi siempre son programas difíciles de leer, con errores difíciles de encontrar. Por eso, todo programa —del más simple al más complejo— conviene diseñarlo siguiendo este proceso:

1. **Entender el problema.** ¿Qué datos entran? ¿Qué resultado se espera? ¿Hay restricciones? Si no puedes explicar el problema en una frase, todavía no lo entiendes lo suficiente para resolverlo.
2. **Diseñar el algoritmo**, en pseudocódigo o diagrama de flujo, **sin pensar todavía en la sintaxis de ningún lenguaje**. Es la parte más importante y la que más se salta quien empieza a programar — el tema de la sección 3.
3. **Traducir el algoritmo a código**, en el lenguaje elegido (hoy, Fortran).
4. **Compilar** — el compilador revisa la sintaxis y, si es correcta, genera un ejecutable.
5. **Probar el programa** con distintos casos, incluyendo casos límite.
6. **Depurar** — corregir los errores que las pruebas revelaron (sección 11).
7. **Documentar** — comentar el código para quien lo lea después, incluido tú mismo en un mes.

Este proceso no es lineal en la práctica —normalmente vuelves atrás entre pasos varias veces— pero **saltarse el paso 2** es, con diferencia, la causa más común de programas que "casi funcionan" pero tienen errores lógicos difíciles de rastrear. Verás este mismo proceso aplicado explícitamente en el ejemplo integrador de la sección 9.

## 3. Diagramas de flujo y pseudocódigo

**¿Para qué sirven?** Permiten **visualizar la estructura lógica de un algoritmo antes de escribir una sola línea de código**, lo que facilita cuatro cosas concretas:

- **Diseñar:** pensar la solución completa sin distraerte con la sintaxis de un lenguaje.
- **Comunicar:** un diagrama se entiende sin saber programar.
- **Implementar:** traducir un diagrama a código es casi mecánico.
- **Depurar:** si el programa falla, comparar su comportamiento contra el diagrama original ayuda a encontrar en qué paso la lógica se desvió.

Un diagrama de flujo usa una simbología estándar:

| Símbolo | Nombre | Función |
|---|---|---|
| Óvalo | Inicio/Fin | Demarca el inicio o fin de un programa o procedimiento |
| Rectángulo | Proceso | Indica las operaciones entre variables e instrucciones para la máquina |
| Rombo | Decisión | Dependiendo del valor de una variable, el flujo del programa se puede bifurcar |
| Paralelogramo | Entrada/Salida | Lectura y escritura de datos |
| Círculo/pantalla | Pantalla | Presentación en pantalla de resultados |
| Flecha | Flujo | Dirección de la secuencia de instrucciones |

El pseudocódigo es la versión en texto de lo mismo: pasos numerados en lenguaje natural estructurado, sin preocuparse todavía por la sintaxis exacta de un lenguaje.

**Ejemplo resuelto 1 — ¿un número es par o impar?** (diagrama de flujo)

```mermaid
flowchart TD
    A([Inicio]) --> B[/Leer numero N/]
    B --> C{N es divisible por 2?}
    C -- Si --> D[Imprimir 'Par']
    C -- No --> E[Imprimir 'Impar']
    D --> F([Fin])
    E --> F
```

**Ejemplo resuelto 2 — promedio de tres notas y si aprueba** (pseudocódigo)

```
Inicio
  Leer nota1, nota2, nota3
  promedio <- (nota1 + nota2 + nota3) / 3
  Si promedio >= 3.0 entonces
      Escribir "Aprobado"
  Sino
      Escribir "Reprobado"
  Fin Si
Fin
```

Ambos ejemplos comparten la misma estructura: **entrada → proceso/decisión → salida**, la que verás en prácticamente todos los programas de este semestre.

**Ejercicio 1 (en clase, con el docente):** en el tablero, elaboren el diagrama de flujo para el algoritmo que resuelve el problema de ir del salón de clase a la Biblioteca Departamental en MIO.

_Espacio para el Ejercicio 1 — inserta una foto del tablero o describe aquí los pasos del diagrama:_

1. …
2. …
3. …

## 4. Estructura de un programa Fortran

Todo programa Fortran se divide en tres secciones:

1. **Sección de declaración** — sentencias no ejecutables al inicio que definen el nombre del programa y el tipo de cada variable.
2. **Sección de ejecución** — las instrucciones que el programa realmente lleva a cabo, en orden.
3. **Sección de terminación** — indica al compilador que el programa terminó.

Veámoslo en el programa más simple posible: lee dos números, los multiplica, e imprime el resultado.

In [ ]:
%%writefile mi_primer_programa.f90
program mi_primer_programa
    implicit none

    integer :: i, j, k       ! los tres son enteros

    print *, "Ingrese los dos numeros a multiplicar: "
    read *, i, j

    k = i * j

    print *, "Resultado = ", k

end program mi_primer_programa

In [ ]:
!gfortran mi_primer_programa.f90 -o mi_primer_programa && ./mi_primer_programa

- **Declaración:** `program mi_primer_programa` nombra el programa; `implicit none` y `integer :: i, j, k` declaran que solo usaremos tres variables, todas enteras.
- **Ejecución:** el `print` pide el dato (un *prompt*), `read` lo recibe, `k = i * j` calcula, y el segundo `print` muestra el resultado. Se ejecutan en orden, de arriba hacia abajo.
- **Terminación:** `end program mi_primer_programa` — debe repetir el nombre dado en `program`. (Fortran también acepta una sentencia `stop` justo antes, pero si `end program` es lo último, `stop` es opcional y casi nunca se escribe explícitamente.)

**Un par de convenciones de estilo.** Este material usa palabras clave en minúscula (`program`, `implicit none`, `if`) porque es lo más común en Fortran moderno; el libro de texto del curso usa mayúsculas para las palabras clave (`PROGRAM`, `IMPLICIT NONE`) y minúsculas para las variables — ambas son válidas, Fortran no distingue mayúsculas de minúsculas. Lo que importa no es cuál elijas, sino que:

<div style="background-color:#eef7ee; border-left:5px solid #4caf50; padding:10px 15px; margin:10px 0;">
<b>Buena práctica de programación:</b> adopta un estilo de escritura y síguelo consistentemente en todos tus programas.
</div>

**Compilar y enlazar.** Antes de correr, un programa Fortran pasa por dos pasos: *compilar* (traducir el texto fuente a código objeto) y *enlazar* (combinarlo con las librerías del sistema para producir un ejecutable) — normalmente en un solo comando, como hace `gfortran archivo.f90 -o ejecutable` en las celdas de este notebook. Los programas pueden correr en modo *batch* (sin interacción, todos los datos ya están en un archivo — típico de programas que corren horas o días) o *interactivo* (el programa pide datos mientras el usuario espera, como todo lo que haremos en este curso).

## 5. Variables, constantes y tipos de datos

Una **variable** se identifica por un nombre y almacena un valor que puede cambiar durante la ejecución. Una **constante** es un valor que no cambia — en Fortran se declara con `parameter`.

**Reglas de nombres:** hasta 63 caracteres, empiezan por una letra, y pueden contener letras, números y guion bajo (`_`). No distinguen mayúsculas de minúsculas. Válidos: `tiempo`, `distancia`, `velocidad_inicial`. Inválidos: `3dias` (empieza con número), `a$` (`$` no es válido).

<div style="background-color:#eef7ee; border-left:5px solid #4caf50; padding:10px 15px; margin:10px 0;">
<b>Buena práctica de programación:</b> usa nombres de variable significativos siempre que sea posible (<code>velocidad_inicial</code>, no <code>v1</code> ni <code>x</code>).
</div>

Tipos de datos básicos:

| Tipo Fortran | Equivalente en C++ | Equivalente en Python | Almacena                           | Ejemplo en Fortran            | Ejemplo en Python   |
| ------------ | ------------------ | --------------------- | ---------------------------------- | ----------------------------- | ------------------- |
| `integer`    | `int`              | `int`                 | Números enteros                    | `integer :: n`                | `n = 10`            |
| `real`       | `double`           | `float`               | Números decimales (punto flotante) | `real :: x`                   | `x = 3.14`          |
| `character`  | `char` / `string`  | `str`                 | Texto                              | `character(len=20) :: nombre` | `nombre = "Carlos"` |
| `logical`    | `bool`             | `bool`                | Verdadero o falso                  | `logical :: encontrado`       | `encontrado = True` |
| `complex`    | `std::complex`     | `complex`             | Números complejos                  | `complex :: z`                | `z = (2.0, 3.0)`    |


<div style="
    border: 2px solid #d62728;
    background-color: #fdecec;
    padding: 12px 16px;
    border-radius: 6px;
    color: #8b0000;
">
**Constantes válidas e inválidas.** Una constante entera no lleva punto decimal ni comas: `0`, `-999`, `+17` son válidas; `1,000,000` (comas) y `-100.` (tiene punto) no lo son. Una constante real sí lleva punto decimal, y opcionalmente un exponente con `E`: `10.`, `-999.9`, `1.0E-3` (= 0.001) son válidas; `111E3` no lo es (falta el punto en la mantisa).
</div>    

```fortran
real, parameter :: pi = 3.14159265
```

**El diccionario de datos.** Es buena práctica documentar, junto a cada declaración, qué representa la variable y en qué unidades — especialmente importante en un curso de física, donde una variable sin unidades es ambigua:

```fortran
real :: temperatura_f   ! Temperatura de entrada (grados Fahrenheit)
real :: temperatura_k   ! Temperatura de salida (kelvin)
```
<div style="background-color:#eef7ee; border-left:5px solid #4caf50; padding:10px 15px; margin:10px 0;">
<b>Buena práctica de programación:</b> crea un diccionario de datos en cada programa — declara y describe cada variable, incluyendo sus unidades físicas cuando aplique.
</div> 

In [ ]:
%%writefile variables.f90
program variables
    implicit none

    integer   :: x = 5       ! Variable entera
    real      :: y           ! Variable de punto flotante
    character :: z = 'A'     ! Variable de un caracter

    y = 3.14

    print *, "x = ", x
    print *, "y = ", y
    print *, "z = ", z

end program variables

In [ ]:
!gfortran variables.f90 -o variables && ./variables

**Recorramos el programa línea por línea:**

- `integer :: x = 5` reserva memoria para un entero llamado `x`, inicializado en 5 en el mismo momento de declararlo.
- `real :: y` reserva espacio para un decimal llamado `y`, sin darle valor todavía.
- `character :: z = 'A'` reserva espacio para un solo caracter, inicializado con comillas simples.
- `y = 3.14` es una **instrucción de asignación**.
- Las tres líneas `print *, ...` imprimen el texto entre comillas seguido del valor de la variable; el asterisco (`*`) le dice a Fortran "usa el formato por defecto".

### Ejercicio 2 — Arreglar el código

El siguiente programa **no compila**. Corre la celda para ver el error de compilación, identifica el problema en las líneas `print` y corrígelo directamente en la celda `%%writefile` de abajo hasta que el programa compile y muestre los tres valores.

In [ ]:
%%writefile ejercicio2_arreglar.f90
program variables_bug
    implicit none

    integer   :: x = 5
    real      :: y
    character :: z = 'A'

    y = 3.14

    print *, "x = " x
    print *, "y = " y
    print *, "z = " z

end program variables_bug

In [ ]:
!gfortran ejercicio2_arreglar.f90 -o ejercicio2 && ./ejercicio2

<div style="background-color:#fdecea; border-left:5px solid #e53935; padding:10px 15px; margin:10px 0;">
<b>Tu diagnóstico:</b> ¿qué error(es) encontraste y cómo lo corregiste?

> …
</div>


## 6. Operaciones aritméticas (y un error clásico: la división entera)

Los operadores aritméticos en Fortran son `+ - * /` y `**` (potencia). La **jerarquía de operaciones** es:

1. Paréntesis primero (de más interno a más externo).
2. Potencias, evaluadas de **derecha a izquierda** (`2.0 ** 3.0 ** 2.0` es `2.0 ** (3.0 ** 2.0)` = $2^9$ = 512, no $(2^3)^2$ = 64 — es la única operación que no va de izquierda a derecha).
3. Multiplicación y división, de izquierda a derecha.
4. Suma y resta, de izquierda a derecha.

**El error más común al empezar en Fortran (o en cualquier lenguaje tipado):** si divides dos `integer`, el resultado es *otro entero*, y la parte decimal se **trunca**, no se redondea. `7 / 2` da `3`, no `3.5`. Si necesitas un resultado decimal, al menos uno de los dos operandos debe ser `real` — Fortran promueve automáticamente la operación completa a punto flotante (*aritmética de modo mixto*).

In [ ]:
%%writefile division_enteros.f90
program division_enteros
    implicit none

    integer :: a, b
    real    :: c, d

    a = 7
    b = 2
    c = 7.0
    d = 2.0

    print *, "7 / 2     (enteros)      = ", a / b
    print *, "7.0 / 2.0 (reales)       = ", c / d
    print *, "7 / 2.0   (modo mixto)   = ", a / d

end program division_enteros

In [ ]:
!gfortran division_enteros.f90 -o division_enteros && ./division_enteros

<div style="background-color:#eef7ee; border-left:5px solid #4caf50; padding:10px 15px; margin:10px 0;">
<b>Buena práctica de programación:</b> no uses aritmética entera para cantidades del mundo real que varían continuamente (distancia, tiempo, temperatura) — resérvala para conteos. Si de verdad necesitas mezclar `integer` y `real`, usa las funciones `real()`, `int()` o `nint()` para dejar la conversión explícita en vez de depender de la promoción automática.
</div>    

## 7. Funciones intrínsecas

Fortran trae incorporadas (sin necesidad de librerías externas) las funciones matemáticas más comunes:

| Función | Hace qué |
|---|---|
| `sqrt(x)` | raíz cuadrada |
| `abs(x)` | valor absoluto |
| `sin(x)`, `cos(x)`, `tan(x)` | trigonométricas — **el argumento debe estar en radianes** |
| `log(x)` | logaritmo natural (base $e$) |
| `mod(a, b)` | residuo de la división entera de `a` entre `b` |
| `max(a, b, ...)`, `min(a, b, ...)` | máximo / mínimo entre varios valores |
| `int(x)`, `real(x)`, `nint(x)` | conversión entre tipos (trunca, convierte, redondea) |

El punto de `sin`/`cos`/`tan` esperando radianes es importante: si tu dato viene en grados, conviértelo primero con $\theta_{rad} = \theta_{grados} \times \pi / 180$.

In [ ]:
%%writefile funciones_intrinsecas.f90
program funciones_intrinsecas
    implicit none

    real :: x, theta_grados, theta_rad
    real, parameter :: pi = 3.14159265

    x = 16.0
    theta_grados = 30.0
    theta_rad = theta_grados * pi / 180.0

    print *, "sqrt(16.0)     = ", sqrt(x)
    print *, "abs(-16.0)     = ", abs(-x)
    print *, "sin(30 grados) = ", sin(theta_rad)
    print *, "cos(30 grados) = ", cos(theta_rad)
    print *, "mod(7, 2)      = ", mod(7, 2)
    print *, "max(3, 9, 5)   = ", max(3, 9, 5)

end program funciones_intrinsecas

In [ ]:
!gfortran funciones_intrinsecas.f90 -o funciones_intrinsecas && ./funciones_intrinsecas

### Ejercicio 3 — Ampliar el código

Modifica `funciones_intrinsecas.f90` (edita la celda `%%writefile` y vuelve a correr las dos celdas) para que además calcule y muestre `tan(theta_rad)`, y verifique la conversión de vuelta: calcula `theta_rad * 180.0 / pi` y comprueba que da de nuevo `30.0`.

## 8. Entrada y salida de datos

La **entrada** es la información que el programa recibe (típicamente del teclado); la **salida** es lo que el programa entrega (típicamente a pantalla). `read *, variable` y `print *, ...` son E/S "dirigida por lista" (*list-directed*): Fortran decide una presentación razonable sin que tú especifiques columnas ni decimales exactos. Veremos formatos personalizados más adelante.

Puedes leer o escribir varios valores en una sola instrucción: `read *, a, b, c`.

In [ ]:
%%writefile entrada_salida.f90
program entrada_salida
    implicit none

    integer :: edad

    print *, "Ingrese su edad: "
    read *, edad

    print *, "En el 2030, usted tendra ", edad + 4, " anios de edad."

end program entrada_salida

In [ ]:
!gfortran entrada_salida.f90 -o entrada_salida && ./entrada_salida

- `read *, edad` **detiene la ejecución del programa** y espera a que el usuario escriba un valor y presione Enter.
- La última línea combina texto literal con una expresión (`edad + 4`) directamente dentro del `print`.

<div style="background-color:#eef7ee; border-left:5px solid #4caf50; padding:10px 15px; margin:10px 0;">
<b>Buena práctica de programación:</b> siempre haz <i>eco</i> de cualquier valor que el usuario ingrese (imprímelo de vuelta) para confirmar que se leyó correctamente, y acompaña toda entrada o salida numérica con sus unidades.
</div>

### Ejercicio 4 — Completar el código

El programa de abajo tiene dos espacios en blanco marcados con `____`. Reemplázalos por la instrucción/valor correcto para que el programa lea la edad y calcule los años que tendrá en 2030 (en este curso, hoy es 2026).

In [ ]:
%%writefile ejercicio4_completar.f90
program completar_edad
    implicit none

    integer :: edad

    print *, "Ingrese su edad: "
    ____ *, edad                 ! (a) instruccion para LEER un valor desde el teclado

    print *, "En el 2030, usted tendra ", edad + ____, " anios de edad."   ! (b) diferencia en anios entre 2026 y 2030

end program completar_edad

In [ ]:
!gfortran ejercicio4_completar.f90 -o ejercicio4 && ./ejercicio4

## 9. Ejemplo integrador: datación por carbono-14

Ahora sí, apliquemos el proceso completo de la sección 2 a un problema físico real, uniendo todo lo visto: variables con diccionario de datos, constantes, funciones intrínsecas, entrada/salida.

**El problema.** Un isótopo radiactivo se desintegra de forma exponencial. Si $Q_0$ es la cantidad inicial de una sustancia radiactiva en $t=0$, la cantidad restante en el tiempo $t$ es

$$Q(t) = Q_0 \, e^{-\lambda t}$$

donde $\lambda$ es la constante de desintegración radiactiva. Los arqueólogos usan esto como un reloj: si conocen el porcentaje de carbono-14 que queda en una muestra orgánica, pueden despejar $t$ y calcular hace cuánto murió el organismo:

$$t_{edad} = -\frac{1}{\lambda}\ln\left(\frac{Q}{Q_0}\right)$$

La constante de desintegración del carbono-14 es $\lambda = 0.00012097\ \text{año}^{-1}$ (bien conocida experimentalmente).

**Paso 1-2 (entender y diseñar):** el programa debe pedir el porcentaje de C-14 remanente, convertirlo a una razón (dividir entre 100), y aplicar la fórmula de arriba usando `log` (logaritmo natural).

In [ ]:
%%writefile datacion_carbono14.f90
program datacion_carbono14
    implicit none

    ! Diccionario de datos: constantes
    real, parameter :: lambda = 0.00012097   ! Constante de decaimiento del C-14 (1/anio)

    ! Diccionario de datos: variables
    real :: porcentaje   ! Porcentaje de C-14 remanente en la muestra (%)
    real :: razon        ! Razon entre el C-14 actual y el original (adimensional)
    real :: edad          ! Edad estimada de la muestra (anios)

    print *, "Ingrese el porcentaje de carbono-14 remanente en la muestra:"
    read *, porcentaje

    print *, "Porcentaje de C-14 remanente: ", porcentaje, " %"

    razon = porcentaje / 100.0
    edad = (-1.0 / lambda) * log(razon)

    print *, "Edad estimada de la muestra: ", edad, " anios"

end program datacion_carbono14

In [ ]:
!gfortran datacion_carbono14.f90 -o datacion_carbono14 && ./datacion_carbono14

**Paso 5 (probar):** si ingresas `50` (la mitad del C-14 original ya se desintegró), el programa debería dar aproximadamente **5729.9 años** — y en efecto, la vida media real del carbono-14, según el CRC Handbook of Chemistry and Physics, es de 5730 años. El resultado del programa coincide con la referencia física real, que es exactamente cómo se valida un programa científico: comparando contra un caso conocido, no solo revisando que "corra sin errores".

Fíjate que este programa ya sigue varias buenas prácticas: usa `implicit none`, tiene diccionario de datos con unidades, usa `parameter` para la constante física en vez de escribir `0.00012097` directamente en la fórmula, y hace eco del valor ingresado.

## 10. Operadores lógicos y decisiones (`if`)

Las operaciones **lógicas** se usan para comparar valores y tomar decisiones dentro del código — le dan a un programa la capacidad de reaccionar distinto según los datos que recibe.

Las variables `logical` solo toman los valores `.TRUE.` o `.FALSE.` (al leerlas del teclado, se aceptan `T` / `F`). Fortran tiene un tipo lógico real — no lo simula con 0/1 como suele hacerse en C++.

| C++ | Fortran | Significado |
|---|---|---|
| `&&` | `.AND.` | Y lógico |
| `\|\|` | `.OR.` | O lógico |
| `!` | `.NOT.` | Negación |
| `==` | `==` (o `.EQ.`) | Igualdad |
| `!=` | `/=` (o `.NE.`) | Distinto |

```fortran
if (condicion) then
    ! instrucciones si la condicion es verdadera
else
    ! instrucciones si es falsa
end if
```

Ejemplo — adaptado del ejercicio de la Lectura 1 (año de nacimiento y si ya cumplió años este año):

In [ ]:
%%writefile operaciones.f90
program operaciones
    implicit none

    integer :: anio_nacimiento, edad
    logical :: ya_cumplio_anios

    print *, "Digite su anio de nacimiento:"
    read *, anio_nacimiento

    print *, "Ya cumplio anios en 2026? (escriba T para Si, F para No)"
    read *, ya_cumplio_anios

    if (ya_cumplio_anios) then
        edad = 2026 - anio_nacimiento
    else
        edad = 2026 - anio_nacimiento - 1
    end if

    print *, "Su edad es ", edad, " anios."

end program operaciones

In [ ]:
!gfortran operaciones.f90 -o operaciones && ./operaciones

`if (ya_cumplio_anios) then` evalúa la variable lógica directamente entre paréntesis — como ya es `logical`, no hace falta comparar nada. Nota que **ambas ramas calculan `edad`, pero con una fórmula distinta**: esa es la esencia de una decisión.

### Ejercicio 5 — Predecir y luego correr

Antes de correr la celda de arriba: si alguien nació en 1998 y **no** ha cumplido años todavía en 2026, ¿qué edad debería mostrar el programa? Escribe tu predicción, luego corre el programa con esos datos y compara.

_**Tu predicción:** …_

_**Resultado real:** …_

## 11. Depuración: cuando el programa no hace lo que esperabas

Vas a pasar, en promedio, más tiempo corrigiendo programas que escribiéndolos — le pasa a cualquiera que programa, no solo a quien está aprendiendo. Los errores (*bugs*) se dividen en tres categorías:

- **Errores de sintaxis:** errores en la sentencia Fortran misma (ortografía, puntuación). Los detecta el compilador, que se niega a generar el ejecutable y señala la línea aproximada — como en el Ejercicio 2. Son los más fáciles de corregir.
- **Errores de ejecución (runtime):** ocurren cuando se intenta una operación matemática ilegal durante la ejecución (por ejemplo, dividir por cero); el programa aborta a mitad de la corrida.
- **Errores lógicos:** el programa compila y corre sin quejarse, pero el resultado es incorrecto. Son los más difíciles de encontrar porque no hay ningún mensaje que te oriente.

El error de programación más común, con diferencia, es el error de tipeo. Si escribiste mal una palabra clave, el compilador lo detecta (error de sintaxis). Pero si confundiste dos nombres de variable *válidos* parecidos (por ejemplo, `vel1` en vez de `vel2`), el compilador no puede ayudarte — eso produce un error lógico silencioso, y es exactamente el tipo de error que `implicit none` no puede prevenir por sí solo.

**Estrategia de depuración**, cuando el programa compila pero da un resultado incorrecto:

1. Revisa primero los datos de entrada — haz eco de todo lo que el usuario ingresa (sección 8) y confirma que es lo que esperabas.
2. Si una expresión es muy larga, divídela en varias asignaciones más pequeñas — son más fáciles de verificar una por una.
3. Revisa la colocación de los paréntesis — es un error común que las operaciones se evalúen en un orden distinto al que pensabas (sección 6). Ante la duda, agrega paréntesis de más.
4. Verifica que todas las variables estén inicializadas antes de usarse.
5. Confirma que las funciones reciban los datos en las unidades correctas — el ejemplo clásico es pasarle grados a `sin`/`cos`/`tan` en vez de radianes (sección 7).
6. Revisa posibles errores por aritmética entera o de modo mixto (sección 6).
7. Si nada de esto revela el problema, agrega `print` temporales en puntos intermedios del programa para ver dónde un valor deja de ser el esperado — y bórralos una vez encuentres el error.
8. Si sigues atascado, explícale el código a otra persona (o a tu docente) en voz alta — es sorprendente cuántas veces uno mismo encuentra el error a mitad de la explicación.

## 12. Buenas prácticas de programación — síntesis

Ya viste estas ideas aplicadas a lo largo del notebook (adaptadas del resumen de buenas prácticas del libro de texto del curso); aquí quedan reunidas como checklist:

1. Usa nombres de variable significativos (sección 5).
2. Usa siempre `implicit none` (sección 4).
3. Crea un diccionario de datos en cada programa: tipo, descripción y unidades de cada variable (sección 5).
4. Usa una cantidad consistente de dígitos en tus constantes (no `3.14` en una parte del programa y `3.14159265` en otra) — nómbralas con `parameter` y referencia el nombre en todas partes (sección 7, sección 9).
5. No uses aritmética entera para cantidades del mundo real que varían continuamente; resérvala para conteos (sección 6).
6. Evita la aritmética de modo mixto salvo en exponenciación; si tienes que mezclar `integer` y `real`, usa `real()`, `int()` o `nint()` para dejar la conversión explícita (sección 6).
7. Usa paréntesis adicionales para que tus expresiones sean más fáciles de leer (sección 6, sección 11).
8. Haz eco de cualquier valor que el usuario ingrese, y acompaña toda entrada/salida con sus unidades (sección 8, sección 9).
9. Inicializa todas las variables antes de usarlas: con una asignación, con `read`, o directamente en la declaración (sección 5, sección 11).
10. Adopta un estilo de programación consistente y síguelo siempre (sección 4).
11. Compila seguido — no esperes a tener 100 líneas para compilar por primera vez (sección 11).
12. Diseña antes de programar (sección 2) — un pseudocódigo o diagrama de flujo de cinco minutos ahorra media hora de depuración después.

## 13. Hoja de referencia rápida

| Concepto | Sintaxis |
|---|---|
| Esqueleto de programa | `program nombre` … `implicit none` … `end program nombre` |
| Declarar entero / real / caracter / lógico | `integer :: n`  ·  `real :: x`  ·  `character :: c`  ·  `logical :: b` |
| Declarar constante | `real, parameter :: pi = 3.14159265` |
| Comentario | `! esto es un comentario` |
| Imprimir | `print *, "texto", variable` |
| Leer | `read *, variable` |
| Decisión | `if (condicion) then` … `else` … `end if` |
| Aritméticos | `+  -  *  /  **` (jerarquía: sección 6) |
| Lógicos | `.AND.  .OR.  .NOT.` |
| Comparación | `==  /=  <  <=  >  >=` |
| Compilar y correr | `gfortran programa.f90 -o programa` luego `./programa` |

**Orden obligatorio de las secciones en un programa Fortran:**

1. `program nombre`
2. `implicit none`
3. Declaraciones de tipo (`real`, `integer`, `character`, `logical` — en cualquier orden entre ellas)
4. Sentencias ejecutables (`=`, `read`, `print`, en el orden que necesite tu algoritmo)
5. `end program nombre`

## 14. Tarea

**1. Pseudocódigo y diagrama de flujo** para cada una de las siguientes tareas (a mano o con Mermaid, como en la sección 3):
   - a. Sacar el libro "Cien años de soledad" de la Biblioteca Central.
   - b. La receta para una tortilla.
   - c. Almorzar en la Cafetería Central.

**2. Diseñe un programa en Fortran** que calcule la altura máxima de un proyectil, conociendo el ángulo respecto a la horizontal y la rapidez inicial. Siga el proceso completo de la sección 2 y el estilo del ejemplo integrador de la sección 9 (diagrama de flujo, diccionario de datos, constantes con `parameter`, eco de entradas).
   - Fórmula: $h_{max} = \dfrac{v_0^2 \sin^2(\theta)}{2g}$, con $g = 9.8\ m/s^2$.
   - Recuerde: `sin` espera el ángulo en radianes (sección 7) — convierta antes de usarlo.
   - Variables sugeridas: `v0`, `angulo_grados`, `angulo_rad`, `g`, `h_max` (todas `real`), y cuidado con no dividir enteros sin querer (sección 6).

**3. Glosario:** defina con sus propias palabras: *algoritmo*, *variable*, *constante*, *tipo de dato*, *diagrama de flujo*, *pseudocódigo*, *entrada de datos*, *salida de datos*, `implicit none`, *diccionario de datos*, *división entera*, *función intrínseca*, *operador lógico*, *error de sintaxis*, *error de ejecución*, *error lógico*.

## 15. Bibliografia

Estas referencias no son necesarias para completar esta sesión ni la tarea.

- Chapman, S. J. *Fortran for Scientists and Engineers* (McGraw-Hill, 2018):
  - Cap. 1 — Introducción a Computadores y el Lenguaje Fortran: §1.1 el computador, §1.2 representación binaria/octal/hexadecimal y tipos de dato en memoria (profundizaremos en esto en la **Semana 2**), §1.3 lenguajes de computador, §1.4–1.5 historia y evolución de Fortran.
  - Cap. 2 — Elementos Básicos de Fortran: §2.4 estructura de un programa, §2.5 constantes y variables, §2.6 aritmética y jerarquía de operaciones, §2.7 funciones intrínsecas, §2.8 E/S dirigida por lista, §2.10 `IMPLICIT NONE`, §2.11 ejemplos resueltos (temperatura, datación por carbono-14), §2.12 depuración de programas, §2.13 resumen y buenas prácticas.
  - §3.1–3.2 Diseño top-down, pseudocódigo y diagramas de flujo.
- Lectura 1: Conceptos Fundamentales en Programación (material del curso, versión C++, adaptada aquí a Fortran).
- Cronograma del curso — Física Computacional 106018C, Semana 1.